<a href="https://colab.research.google.com/github/samikshasali/Chatbot-using-Transformers/blob/main/Chatbot_using_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot using Transformers

## Objective
To build an interactive chatbot using Hugging Face Transformers (DialoGPT) that can generate meaningful responses.

## Internship
Data Science Internship – February 2026 at Innomatics Research Labs

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

In [ ]:
fact_knowledge = {
    "who created python": "Python was created by Guido van Rossum and released in 1991.",
    "what is python": "Python is a high-level, interpreted programming language created by Guido van Rossum.",
    "what is artificial intelligence": "Artificial Intelligence refers to the simulation of human intelligence by machines.",
    "who is guido van rossum": "Guido van Rossum is the creator of Python programming language.",
    "what is machine learning": "Machine Learning is a subset of AI that enables systems to learn from data."
}

In [ ]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("User: ").strip()

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break

    lower_input = user_input.lower()
    response = None

    for key in fact_knowledge:
        if key in lower_input:
            response = fact_knowledge[key]
            break

    if response is None:
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=torch.ones_like(bot_input_ids),
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7
        )

        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    print("Chatbot:", response)